# **2.3. Practice: Fine-Tuning**

## **0. Import Library**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.metrics import *

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## **1. Load dataset**

* select the provided [company-document-text.csv] file.

In [ ]:
uploaded = files.upload()
data_path = list(uploaded.keys())[0]

In [ ]:
data_path

In [ ]:
data = pd.read_csv(data_path)
data.head()

In [ ]:
sns.countplot(x='label', data = data, hue = 'label')
plt.grid()
plt.show()

In [ ]:
sns.histplot(x='word_count', data = data, bins = 50)
plt.grid()
plt.show()

## **2. Pre-processing Data**

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['label'] = le.fit_transform(data['label'])
data.head()

In [ ]:
data2 = data.loc[:, ['text','label']].copy()

In [ ]:
train, val = train_test_split(data2, test_size=0.3, random_state=10)

In [ ]:
# tensor dataset
train_ds = Dataset.from_pandas(train)
val_ds = Dataset.from_pandas(val)

In [ ]:
train_ds[:3]

## **3. Tokenization**

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def preprocess_function(data):
    return tokenizer(data['text'], padding="max_length", truncation=True, max_length=100)

train_ds = train_ds.map(preprocess_function, batched=True)
val_ds = val_ds.map(preprocess_function, batched=True)

## **4. Load Pre-trained Model**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 4)

## **5. Set Training Arguments**

In [ ]:
args = TrainingArguments(
    output_dir = './results',          # where to save checkpoints/outputs
    eval_strategy = "epoch",           # run eval at the end of each epoch
    save_strategy = "epoch",           # save a checkpoint every epoch
    learning_rate = 2e-5,              # base learning rate
    per_device_train_batch_size = 32,  # train batch size per device
    per_device_eval_batch_size = 32,   # eval batch size per device
    num_train_epochs  = 3,             # total training epochs
    weight_decay = 0.02,               # L2 weight decay strength
    load_best_model_at_end = True,     # reload best checkpoint (by metric)
    logging_dir ='./logs',             # TensorBoard log directory
    logging_steps = 10,                # log every N training steps
    report_to="tensorboard"            # send logs to TensorBoard
)

## **6. Fine-tune Model**

In [ ]:
trainer = Trainer(
    model = model,                # Model to be trained
    args = args,                  # Training settings and hyperparameters
    train_dataset = train_ds,     # Training data
    eval_dataset = val_ds,        # Validation data
    tokenizer = tokenizer,        # Tokenizer used for preprocessing
)


In [ ]:
trainer.train()

## **7. Evaluate**

In [ ]:
want_keys = ["input_ids", "attention_mask"]              # keys the model expects (add "token_type_ids" for BERT if present)
batch = {k: torch.tensor(val_ds[k]) for k in want_keys}  # convert dataset columns (lists) into torch tensors
batch = {k: v.to(device) for k, v in batch.items()}      # move tensors to the target device (CPU/GPU)

with torch.no_grad():                                    # disable gradient tracking for inference
    outputs = model(**batch)                             # forward pass; batch must be a dict of tensors
logits = outputs.logits                                  # model scores, shape: [N, num_labels]
pred  = logits.argmax(dim=-1)                            # predicted class ids (argmax over label dimension)

In [ ]:
pred = pred.cpu().numpy()
print(confusion_matrix(val_ds['label'], pred))
print(classification_report(val_ds['label'], pred, target_names = le.classes_))

## **8. Save the model**
* Create a folder
* Save the model and tokenizer to the specified path

In [ ]:
fine_tuned_path = 'model'              # Set the folder name for saving the model and tokenizer
model.save_pretrained(fine_tuned_path) # Save the trained model to the specified directory
tokenizer.save_pretrained(fine_tuned_path) # Save the tokenizer to the same folder

* Load the saved model and tokenizer.

In [ ]:
model2 = AutoModelForSequenceClassification.from_pretrained(fine_tuned_path)
tokenizer2 = AutoTokenizer.from_pretrained(fine_tuned_path)

* Let's use the loaded model and tokenizer.

In [ ]:
sentence = 'mario pontes products product id  product  quantity  unit price  10 ikura 70 31 31 gorgonzola telino 35 12 5 77 original frankfurter grüne soße 40 13 page 1'
inputs = tokenizer2(sentence, return_tensors="pt")  # Tokenize input sentence and convert to PyTorch tensors

outputs = model2(**inputs)  # Perform forward pass with the model
logits = outputs.logits  # Retrieve raw prediction scores (logits)

prob = logits.softmax(dim=-1) # Convert logits to probabilities using softmax
pred = prob.argmax(dim=-1).item() # Get the index of the class with the highest probability

label_list = le.classes_  # List of class names from label encoder

print("Predicted class index:", pred)  # Print predicted class index
print("Predicted class name:", label_list[pred])  # Print predicted class label name
print("Probabilities:", prob.tolist()[0])  # Print predicted class probabilities